# 3장 06. Kubernetes MLflow manifest 확인

이 notebook은 Kubernetes에 MLflow를 먼저 배포하는 선언이 있는지 확인합니다. 실제 cluster에 적용하는 작업은 Argo CD와 `kubectl`이 준비된 VM에서 script로 진행하고, 여기서는 manifest의 핵심 항목을 읽습니다.


## 기본 개념: MLflow를 먼저 배포하는 이유

KServe는 모델 endpoint를 만들지만, 어떤 모델을 후보로 올렸는지는 MLflow 기록과 model artifact 경로가 설명합니다. 그래서 강의 흐름은 다음 순서가 자연스럽습니다.

1. MLflow server를 container로 띄워 실험과 모델 artifact를 기록합니다.
2. Kubernetes에도 MLflow Deployment/Service/PVC를 선언합니다.
3. Argo CD가 MLflow manifest를 먼저 sync합니다.
4. 그 다음 KServe InferenceService가 candidate model artifact를 endpoint로 선언합니다.


## 1. manifest 파일 찾기

로컬 Jupyter와 JupyterLite에서 모두 실행되도록 여러 후보 경로 중 실제 존재하는 파일을 찾습니다.


In [1]:
from pathlib import Path

candidates = [
    Path("gitops/base/mlflow-tracking.yaml"),
    Path("../../gitops/base/mlflow-tracking.yaml"),
    Path("artifacts/deployment/chapter_03/gitops/base/mlflow-tracking.yaml"),
    Path("../artifacts/deployment/chapter_03/gitops/base/mlflow-tracking.yaml"),
]

manifest_path = next((path for path in candidates if path.exists()), None)
print(f"manifest_path={manifest_path}")
assert manifest_path is not None, "MLflow Kubernetes manifest를 찾지 못했습니다."
manifest_text = manifest_path.read_text(encoding="utf-8")


manifest_path=../../gitops/base/mlflow-tracking.yaml


## 2. 핵심 Kubernetes object 확인

MLflow 배포에 필요한 최소 object는 PVC, Deployment, Service입니다. PVC는 artifact 저장 위치, Deployment는 MLflow server 실행, Service는 cluster 내부 접근 경로를 뜻합니다.


In [2]:
import re
import pandas as pd

objects = []
for kind in ["PersistentVolumeClaim", "Deployment", "Service"]:
    matched = re.search(rf"kind:\s*{kind}.*?name:\s*([^\n]+)", manifest_text, flags=re.S)
    objects.append({"kind": kind, "name": matched.group(1).strip() if matched else "missing"})

pd.DataFrame(objects)


,kind,name
0,PersistentVolumeClaim,ai-quality-models
1,Deployment,mlflow-tracking
2,Service,mlflow-tracking


출력에서 세 object가 모두 보여야 Kubernetes에 MLflow를 먼저 띄우는 배포 의도가 있다고 말할 수 있습니다. 다만 manifest 확인은 live 배포 확인이 아닙니다. Argo CD sync와 Pod Ready 확인은 별도입니다.


## 3. sync 순서와 artifact 위치 확인

Argo CD sync wave와 artifact root를 확인합니다. MLflow가 먼저 적용되고 KServe가 뒤에 적용되어야 모델 저장소 경로를 설명하기 쉽습니다.


In [3]:
sync_waves = re.findall(r'argocd.argoproj.io/sync-wave:\s*"?([^"\n]+)"?', manifest_text)
artifact_root = re.search(r"--default-artifact-root\n\s*-\s*([^\n]+)", manifest_text)

print(f"sync_waves={sync_waves}")
print(f"artifact_root={artifact_root.group(1).strip() if artifact_root else 'missing'}")
print("mlflow_live_check=requires argocd sync and kubectl rollout/status")


sync_waves=['0', '1', '2']
artifact_root=file:///mlflow/artifacts
mlflow_live_check=requires argocd sync and kubectl rollout/status


이 결과는 다음 단계의 KServe manifest와 연결됩니다. MLflow가 candidate model artifact를 남기고, KServe는 `storageUri`로 해당 artifact 위치를 읽는 구조여야 합니다.
